# Bayesian Structural Time Series: CausalImpact

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [ ]:
"""Shared setup for all chapters: imports, plotting theme, and data generators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # which markets get the program
    treat_week=60,              # program launch week
    true_effect=8.0,            # true lift in weekly revenue ($k)
    effect_ramp=6,              # weeks for the effect to reach full size
    seed=42,
):
    """NorthShore Retail store-week panel with known treatment effect."""
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets
            post = w >= treat_week
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])


def make_customers(n=8000, true_att=12.0, seed=42):
    """Loyalty-program adoption data with selection on observables."""
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)
    spend0 = rng.lognormal(4.0, 0.5, n)
    engage = rng.beta(2, 5, n) * 10
    urban = rng.binomial(1, 0.55, n)
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))


panel = make_panel()
cust = make_customers()
X_cols = ["tenure", "spend0", "engage", "urban"]
TRUE_ATT = 12.0


CausalImpact (Brodersen et al. 2015, Google) generalizes ITS by building a **Bayesian structural time-series (BSTS)** model of the treated series using (a) its own trend/seasonality and (b) **contemporaneous control series** that were not affected by the intervention. It then forecasts the counterfactual into the post-period and reports the difference with full posterior uncertainty — including a cumulative-effect estimate, which is exactly what finance stakeholders want ("how many incremental dollars?").

**Identifying assumptions:** (1) the control series are predictive of the treated series and (2) the controls were *not themselves affected* by the intervention (no spillovers), and (3) the pre-period relationship between controls and the treated series persists post-period.

**Business setting:** Fresh Local launched in market 0 only. Markets 4–11 are clean controls.

In [ ]:
%pip install tfp-causalimpact
import causalimpact

mkt = (panel.groupby(["week", "market"]).revenue.mean()
            .unstack("market"))
y = mkt[0]                                  # treated market
X = mkt[[4, 5, 6, 7, 8, 9, 10, 11]]        # never-treated controls
data = pd.concat([y.rename("y"), X], axis=1)

pre_period = [0, 59]
post_period = [60, 103]

impact = causalimpact.fit_causalimpact(
    data, pre_period=pre_period, post_period=post_period)
print(causalimpact.summary(impact, output_format="summary"))
causalimpact.plot(impact)

The summary reports:

- **Average effect** per post-period time point with a credible interval,
- **Cumulative effect** (the headline incremental-revenue number),
- A **posterior tail probability** (a Bayesian analogue of a p-value).

The three-panel plot shows: observed vs. counterfactual, pointwise effect, and cumulative effect.

## Practical guidance

- **Control selection is the whole game.** Good controls: sibling markets, unexposed product categories, industry indices, competitor traffic. Run the "could this control plausibly have been affected?" test on every candidate. Spike-and-slab priors inside BSTS handle *many* candidate controls by selecting among them, but they cannot fix a contaminated control.
- **Placebo-in-space:** rerun the analysis pretending each control market was treated; the distribution of placebo "effects" calibrates how unusual the real one is.
- **Placebo-in-time:** end the data before the true launch and impose a fake launch date; you should find nothing.
- **When to prefer CausalImpact over DiD:** one (or few) treated units, rich donor series, and doubt about strict parallel trends — BSTS only requires a stable *predictive* relationship, not parallel level-trends. When you have many treated and control units, panel DiD uses the data more fully.